# Character Abstraction: Bimodal Networks & Quote Scoring

For each BookNLP-parsed + LLM-resolved text:
1. Score character **modifiers**, **agent/patient verbs**, and **possessives** for abstractness
2. Score character **quotes** for abstractness
3. Build bimodal character ↔ abstract/concrete descriptor networks

In [ ]:
import os, sys, json, glob
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
sys.path.insert(0, os.path.expanduser('~/github/lltk'))
sys.path.insert(0, os.path.expanduser('~/github/abslithists/abstraction'))

from abstraction.norms import get_orignorms
from abstraction.tokenize import tokenize

In [ ]:
# Load allnorms (includes period-specific vec norms)
from abstraction.norms import get_allnorms
norms = get_allnorms(remove_stopwords=False)

# Period-specific median columns
PERIOD_COLS = {
    'C16': 'Abs-Conc.Median.C16',
    'C17': 'Abs-Conc.Median.C17',
    'C18': 'Abs-Conc.Median.C18',
    'C19': 'Abs-Conc.Median.C19',
    'C20': 'Abs-Conc.Median.C20',
    'C21': 'Abs-Conc.Median.C21',
}
FALLBACK_COL = 'Abs-Conc.Median.median'

def year_to_period(year):
    if year is None or pd.isna(year):
        return None
    y = int(year)
    if y < 1600: return 'C16'
    elif y < 1700: return 'C17'
    elif y < 1800: return 'C18'
    elif y < 1900: return 'C19'
    elif y < 2000: return 'C20'
    else: return 'C21'

def get_norm_col(year):
    """Get the period-specific norm column for a given year."""
    period = year_to_period(year)
    if period and period in PERIOD_COLS:
        return PERIOD_COLS[period]
    return FALLBACK_COL

print(f'Norms: {len(norms)} words')
for p, col in PERIOD_COLS.items():
    print(f'  {p}: {norms[col].notna().sum()} words')

In [ ]:
# Find all texts with BookNLP + resolved characters
resolved_paths = glob.glob(os.path.expanduser(
    '~/lltk_data/corpora/*/booknlp/en_small/*/characters_resolved.json'
))
print(f'{len(resolved_paths)} texts with resolved characters')

# Build inventory
texts_info = []
for p in sorted(resolved_paths):
    parts = p.split('/corpora/')[1].split('/booknlp/')
    corpus = parts[0]
    text_id = parts[1].replace('en_small/', '').replace('/characters_resolved.json', '')
    booknlp_dir = os.path.dirname(p)
    
    # Check required files exist
    has_book = os.path.exists(os.path.join(booknlp_dir, 'text.book'))
    has_quotes = os.path.exists(os.path.join(booknlp_dir, 'text.quotes'))
    has_tokens = os.path.exists(os.path.join(booknlp_dir, 'text.tokens'))
    
    if has_book and has_quotes and has_tokens:
        texts_info.append({
            'corpus': corpus, 'text_id': text_id,
            '_id': f'_{corpus}/{text_id}',
            'booknlp_dir': booknlp_dir,
        })

texts_df = pd.DataFrame(texts_info)
print(f'{len(texts_df)} texts with all required files')
print(texts_df.corpus.value_counts().to_string())

In [ ]:
# Get metadata (title, author, year) from LLTK
import lltk

def get_text_meta(_id):
    """Get title/author/year from LLTK DB."""
    try:
        row = lltk.db.get(_id)
        if row:
            return row.get('title', ''), row.get('author', ''), row.get('year')
    except:
        pass
    return '', '', None

for i, row in texts_df.iterrows():
    title, author, year = get_text_meta(row['_id'])
    texts_df.loc[i, 'title'] = title
    texts_df.loc[i, 'author'] = author
    texts_df.loc[i, 'year'] = year

texts_df['year'] = pd.to_numeric(texts_df['year'], errors='coerce')
texts_df[['_id', 'title', 'author', 'year']].sort_values('year')

## Extract character descriptors and quotes

For each text, extract:
- **Modifiers** (adj/noun applied to character): from `text.book` JSON `mod` field
- **Agent verbs** (character as subject): from `text.book` JSON `agent` field
- **Patient verbs** (character as object): from `text.book` JSON `patient` field  
- **Possessives** (nouns possessed): from `text.book` JSON `poss` field
- **Quotes** (character speech): from `text.quotes` TSV, tokens looked up in `text.tokens`

Each word is scored using `Abs-Conc.Median` from orignorms.

In [ ]:
def score_word(word, col, norms_df=norms):
    """Return concreteness z-score for a word using given norm column, fallback to median."""
    w = word.lower().strip()
    if w in norms_df.index:
        v = norms_df.loc[w, col]
        if pd.notna(v):
            return v
        v = norms_df.loc[w, FALLBACK_COL]
        if pd.notna(v):
            return v
    return np.nan

def load_resolved_chars(booknlp_dir):
    """Load resolved characters, return {char_id: char_info} mapping.
    Only keeps named characters (type='character', proper-noun name)."""
    with open(os.path.join(booknlp_dir, 'characters_resolved.json')) as f:
        data = json.load(f)
    chars = data if isinstance(data, list) else data.get('characters', [])
    
    # Filter: only type='character', exclude collectives/abstractions/noise
    # Also exclude generic unnamed characters (starts with lowercase, "the", "a", "his", etc.)
    GENERIC_PREFIXES = ('the ', 'a ', 'an ', 'his ', 'her ', 'my ', 'our ', 'any ', 'some ',
                        'this ', 'that ', 'unnamed', 'unknown', 'generic', 'people', 'persons')
    
    def is_named(c):
        if c.get('type') not in ('character',):
            return False
        name = c.get('name', '').strip()
        if not name:
            return False
        name_lower = name.lower()
        if any(name_lower.startswith(p) for p in GENERIC_PREFIXES):
            return False
        # Must start with uppercase (proper noun)
        if not name[0].isupper():
            return False
        return True
    
    id_to_char = {}
    named_chars = []
    for c in chars:
        if not is_named(c):
            continue
        named_chars.append(c)
        for cid in c.get('ids', []):
            if cid not in id_to_char:
                id_to_char[cid] = c
    return named_chars, id_to_char

def load_book_json(booknlp_dir):
    with open(os.path.join(booknlp_dir, 'text.book')) as f:
        return json.load(f)['characters']

def _resolve_char_id(raw_id, id_to_char):
    for fmt in [f"C{raw_id:02d}", f"C{raw_id:03d}"]:
        if fmt in id_to_char:
            return fmt, id_to_char[fmt]
    return None, None

def extract_char_descriptors(booknlp_dir, year=None):
    """Extract scored descriptors for each named character."""
    col = get_norm_col(year)
    chars, id_to_char = load_resolved_chars(booknlp_dir)
    raw_chars = load_book_json(booknlp_dir)
    
    rows = []
    for rc in raw_chars:
        char_id, resolved = _resolve_char_id(rc['id'], id_to_char)
        if not resolved:
            continue
        
        for role in ['mod', 'agent', 'patient', 'poss']:
            for item in rc.get(role, []):
                word = item['w']
                rows.append({
                    'char_name': resolved['name'],
                    'char_id': char_id,
                    'char_gender': resolved.get('gender', 'unknown'),
                    'role': role, 'word': word.lower(),
                    'score': score_word(word, col),
                    'token_i': item['i'],
                })
    
    return pd.DataFrame(rows)

# Test
test_dir = texts_df.iloc[0]['booknlp_dir']
test_year = texts_df.iloc[0]['year']
test_desc = extract_char_descriptors(test_dir, year=test_year)
print(f"Test: {texts_df.iloc[0]['_id']} (year={test_year})")
print(f"  {len(test_desc)} descriptor rows, {test_desc.score.notna().sum()} scored")
print(f"  Named characters: {sorted(test_desc.char_name.unique())}")

In [ ]:
def extract_char_quotes(booknlp_dir, year=None):
    """Extract quotes per named character, score each quote's words."""
    col = get_norm_col(year)
    chars, id_to_char = load_resolved_chars(booknlp_dir)
    
    quotes_df = pd.read_csv(os.path.join(booknlp_dir, 'text.quotes'), sep='\t')
    tokens_df = pd.read_csv(os.path.join(booknlp_dir, 'text.tokens'), sep='\t')
    
    SKIP_POS = {'DT', 'IN', 'CC', 'TO', 'PRP', 'PRP$', 'WP', 'WP$', 'WDT', 'WRB',
                'MD', 'EX', 'PDT', 'RP', 'UH', '.', ',', ':', '``', "''", '-LRB-', '-RRB-',
                'CD', 'SYM', 'LS', 'FW', 'XX'}
    
    rows = []
    for _, q in quotes_df.iterrows():
        char_id, resolved = _resolve_char_id(int(q['char_id']), id_to_char)
        if not resolved:
            continue
        
        start, end = int(q['quote_start']), int(q['quote_end'])
        quote_toks = tokens_df[
            (tokens_df['token_ID_within_document'] >= start) &
            (tokens_df['token_ID_within_document'] <= end)
        ]
        
        words, scores = [], []
        for _, tok in quote_toks.iterrows():
            pos = str(tok.get('fine_POS_tag', ''))
            if pos in SKIP_POS:
                continue
            w = str(tok['lemma']).lower()
            s = score_word(w, col)
            if pd.notna(s):
                words.append(w)
                scores.append(s)
        
        if scores:
            rows.append({
                'char_name': resolved['name'],
                'char_id': char_id,
                'char_gender': resolved.get('gender', 'unknown'),
                'quote_text': q.get('quote', ''),
                'quote_start': start,
                'n_scored_words': len(scores),
                'mean_score': np.mean(scores),
            })
    
    return pd.DataFrame(rows)

# Test
test_quotes = extract_char_quotes(test_dir, year=test_year)
print(f"Test quotes: {len(test_quotes)} scored quotes, named characters only")
if len(test_quotes):
    print(test_quotes.groupby('char_name').agg(
        n_quotes=('mean_score', 'count'),
        mean_abs=('mean_score', 'mean'),
    ).sort_values('n_quotes', ascending=False).head(10).round(3))

## Process all texts

In [ ]:
MIN_YEAR = 1470

# Load text-level abstraction scores from data/scores/v8/
scores_dir = os.path.expanduser('~/github/abslithists/abstraction/data/scores/v8')
text_scores = {}
for csv_path in glob.glob(os.path.join(scores_dir, '*.csv')):
    corpus_name = os.path.basename(csv_path).replace('.csv', '')
    try:
        sdf = pd.read_csv(csv_path)
        for _, row in sdf.iterrows():
            _id = f'_{corpus_name}/{row["id"]}'
            # Collect period-specific median scores
            score_cols = {c: row[c] for c in sdf.columns if 'Abs-Conc.Median' in c and pd.notna(row.get(c))}
            text_scores[_id] = score_cols
    except:
        pass
print(f"Loaded text-level scores for {len(text_scores)} texts")

# Filter texts to year >= 1470
texts_post = texts_df[texts_df.year >= MIN_YEAR].copy()
print(f"Texts with year >= {MIN_YEAR}: {len(texts_post)}")

# Process
all_descriptors = []
all_quotes = []
errors = []

for i, row in texts_post.iterrows():
    try:
        desc = extract_char_descriptors(row['booknlp_dir'], year=row['year'])
        if len(desc):
            desc['_id'] = row['_id']
            desc['year'] = row['year']
            desc['title'] = row['title']
            all_descriptors.append(desc)
        
        quotes = extract_char_quotes(row['booknlp_dir'], year=row['year'])
        if len(quotes):
            quotes['_id'] = row['_id']
            quotes['year'] = row['year']
            quotes['title'] = row['title']
            all_quotes.append(quotes)
    except Exception as e:
        errors.append((row['_id'], str(e)))

desc_df = pd.concat(all_descriptors, ignore_index=True) if all_descriptors else pd.DataFrame()
quotes_df = pd.concat(all_quotes, ignore_index=True) if all_quotes else pd.DataFrame()

print(f"\nDescriptors: {len(desc_df)} rows across {desc_df._id.nunique()} texts")
print(f"  Scored: {desc_df.score.notna().sum()} ({desc_df.score.notna().mean():.0%})")
print(f"  Named characters: {desc_df.char_name.nunique()}")
print(f"Quotes: {len(quotes_df)} scored quotes across {quotes_df._id.nunique() if len(quotes_df) else 0} texts")
if errors:
    print(f"Errors: {len(errors)}")
    for _id, e in errors[:5]:
        print(f"  {_id}: {e}")

## Character-level abstractness profiles

Per-character mean abstractness of descriptors (mod, agent, patient, poss) and quotes.

In [ ]:
# Per-character descriptor abstractness by role
scored = desc_df.dropna(subset=['score'])
char_role = scored.groupby(['_id', 'title', 'year', 'char_name', 'char_gender', 'role']).agg(
    mean_score=('score', 'mean'),
    n_words=('score', 'count'),
).reset_index()

# Per-character overall descriptor score (all roles)
char_overall = scored.groupby(['_id', 'title', 'year', 'char_name', 'char_gender']).agg(
    desc_mean=('score', 'mean'),
    desc_n=('score', 'count'),
).reset_index()

# Per-character quote score
if len(quotes_df):
    char_quotes_agg = quotes_df.groupby(['_id', 'title', 'year', 'char_name', 'char_gender']).agg(
        quote_mean=('mean_score', 'mean'),
        quote_n=('n_scored_words', 'sum'),
        n_quotes=('mean_score', 'count'),
    ).reset_index()
    
    # Merge
    char_profiles = char_overall.merge(char_quotes_agg, 
        on=['_id', 'title', 'year', 'char_name', 'char_gender'], how='outer')
else:
    char_profiles = char_overall
    char_profiles['quote_mean'] = np.nan
    char_profiles['quote_n'] = 0
    char_profiles['n_quotes'] = 0

# Filter to characters with enough data
MIN_WORDS = 5
char_profiles_filt = char_profiles[
    (char_profiles.desc_n >= MIN_WORDS) | (char_profiles.quote_n >= MIN_WORDS)
].copy()

print(f"{len(char_profiles_filt)} characters with >= {MIN_WORDS} scored words")
print(f"From {char_profiles_filt._id.nunique()} texts")
print()

# Show most abstract and most concrete characters
print("=== Most ABSTRACT characters (by descriptors) ===")
top_abs = char_profiles_filt.nsmallest(15, 'desc_mean')
for _, r in top_abs.iterrows():
    print(f"  {r.desc_mean:+.3f}  {r.char_name:<25s} ({r.title[:40]}, {r.year:.0f})")

print()
print("=== Most CONCRETE characters (by descriptors) ===")
top_conc = char_profiles_filt.nlargest(15, 'desc_mean')
for _, r in top_conc.iterrows():
    print(f"  {r.desc_mean:+.3f}  {r.char_name:<25s} ({r.title[:40]}, {r.year:.0f})")

## Historical trend: character abstractness over time

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Descriptor abstractness by role over time
for role, color in [('mod', 'tab:blue'), ('agent', 'tab:orange'), ('patient', 'tab:green'), ('poss', 'tab:red')]:
    sub = char_role[char_role.role == role].groupby('year').mean_score.mean()
    if len(sub) > 3:
        axes[0].scatter(sub.index, sub.values, alpha=0.5, s=20, color=color, label=role)
        s = sub.sort_index()
        if len(s) > 5:
            axes[0].plot(s.rolling(5, center=True, min_periods=2).mean(), color=color, lw=2)
axes[0].set_title('Descriptor abstractness by role')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Concreteness (- = abstract, + = concrete)')
axes[0].legend()
axes[0].axhline(0, color='grey', lw=0.5, ls='--')
axes[0].set_xlim(MIN_YEAR - 10, 2030)

# 2. Overall descriptor vs quote abstractness per text
text_desc = char_profiles_filt.groupby(['_id', 'year']).agg(
    desc=('desc_mean', 'mean'), quote=('quote_mean', 'mean')
).reset_index().dropna(subset=['year'])

for measure, color, label in [('desc', 'tab:blue', 'Descriptors'), ('quote', 'tab:red', 'Quotes')]:
    sub = text_desc.dropna(subset=[measure]).sort_values('year')
    axes[1].scatter(sub.year, sub[measure], alpha=0.5, s=25, color=color, label=label)
    if len(sub) > 5:
        r = sub.set_index('year')[measure].rolling(5, center=True, min_periods=2).mean()
        axes[1].plot(r.index, r.values, color=color, lw=2)
axes[1].set_title('Character abstractness: descriptors vs quotes')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Concreteness')
axes[1].legend()
axes[1].axhline(0, color='grey', lw=0.5, ls='--')
axes[1].set_xlim(MIN_YEAR - 10, 2030)

# 3. Descriptor score by gender over time
for gender, color in [('male', 'tab:blue'), ('female', 'tab:red')]:
    sub = char_profiles_filt[char_profiles_filt.char_gender == gender]
    by_year = sub.groupby('year').desc_mean.mean().sort_index()
    if len(by_year) > 3:
        axes[2].scatter(by_year.index, by_year.values, alpha=0.4, s=20, color=color, label=gender)
        if len(by_year) > 5:
            axes[2].plot(by_year.rolling(5, center=True, min_periods=2).mean(), color=color, lw=2)
axes[2].set_title('Character abstractness by gender')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Concreteness')
axes[2].legend()
axes[2].axhline(0, color='grey', lw=0.5, ls='--')
axes[2].set_xlim(MIN_YEAR - 10, 2030)

plt.tight_layout()
plt.show()

## Bimodal character ↔ descriptor networks

Bipartite graphs: named characters (center) ↔ abstract words (left, red) / concrete words (right, green). Separate plots for each descriptor role: **mod** (adjectives), **agent** (verbs as subject), **patient** (verbs as object), **poss** (possessives).

In [ ]:
import networkx as nx

def build_bimodal_network(desc_df_text, min_char_words=3, zcut=0.5, top_words=10):
    """Build bipartite character ↔ descriptor network for one text.
    Keep only top N abstract + top N concrete words per character.
    """
    scored = desc_df_text.dropna(subset=['score'])
    char_counts = scored.groupby('char_name').size()
    keep_chars = char_counts[char_counts >= min_char_words].index
    scored = scored[scored.char_name.isin(keep_chars)]
    if len(scored) == 0:
        return None
    
    edges = []
    for char_name, grp in scored.groupby('char_name'):
        word_stats = grp.groupby('word').agg(
            freq=('score', 'count'), mean_score=('score', 'mean')
        ).reset_index()
        abs_words = word_stats[word_stats.mean_score <= -zcut].nlargest(top_words, 'freq')
        conc_words = word_stats[word_stats.mean_score >= zcut].nlargest(top_words, 'freq')
        for _, row in pd.concat([abs_words, conc_words]).iterrows():
            edges.append({'char_name': char_name, 'word': row.word,
                          'weight': row.freq, 'score': row.mean_score})
    
    if not edges:
        return None
    
    edge_df = pd.DataFrame(edges)
    G = nx.Graph()
    for char_name in edge_df.char_name.unique():
        char_scores = scored[scored.char_name == char_name].score
        G.add_node(char_name, bipartite=0, node_type='character',
                   mean_score=char_scores.mean(), n_words=len(char_scores))
    for _, row in edge_df.iterrows():
        word = row['word']
        if word not in G:
            wclass = 'abstract' if row['score'] <= -zcut else 'concrete'
            G.add_node(word, bipartite=1, node_type='word',
                       word_class=wclass, score=row['score'])
        G.add_edge(row['char_name'], word, weight=row['weight'])
    return G

def plot_bimodal_network(G, title='', ax=None):
    if G is None or len(G) == 0:
        if ax: ax.set_title(f'{title}\n(no data)'); ax.axis('off')
        return
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    chars = sorted([n for n, d in G.nodes(data=True) if d.get('node_type') == 'character'],
                   key=lambda n: G.nodes[n].get('mean_score', 0))
    abs_words = sorted([n for n, d in G.nodes(data=True) if d.get('word_class') == 'abstract'],
                       key=lambda n: G.nodes[n].get('score', 0))
    conc_words = sorted([n for n, d in G.nodes(data=True) if d.get('word_class') == 'concrete'],
                        key=lambda n: G.nodes[n].get('score', 0))
    
    pos = {}
    for i, c in enumerate(chars):
        pos[c] = (0, (i - len(chars)/2) * 1.5)
    for i, w in enumerate(abs_words):
        pos[w] = (-2, (i - len(abs_words)/2) * 0.8)
    for i, w in enumerate(conc_words):
        pos[w] = (2, (i - len(conc_words)/2) * 0.8)
    
    char_scores = [G.nodes[c].get('mean_score', 0) for c in chars]
    char_sizes = [min(G.nodes[c].get('n_words', 10), 200) * 8 for c in chars]
    nx.draw_networkx_nodes(G, pos, nodelist=chars, node_color=char_scores,
                           cmap=plt.cm.RdBu_r, node_size=char_sizes,
                           vmin=-1.5, vmax=1.5, ax=ax, node_shape='s', edgecolors='black')
    if abs_words:
        nx.draw_networkx_nodes(G, pos, nodelist=abs_words, node_color='#d62728',
                               node_size=80, ax=ax, alpha=0.7)
    if conc_words:
        nx.draw_networkx_nodes(G, pos, nodelist=conc_words, node_color='#2ca02c',
                               node_size=80, ax=ax, alpha=0.7)
    
    for u, v, d in G.edges(data=True):
        word_node = v if G.nodes[v].get('node_type') == 'word' else u
        color = '#d62728' if G.nodes[word_node].get('word_class') == 'abstract' else '#2ca02c'
        ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]],
                color=color, alpha=0.25, lw=max(0.5, d['weight'] / 3))
    
    nx.draw_networkx_labels(G, pos, {n: n for n in chars}, font_size=9, font_weight='bold', ax=ax)
    nx.draw_networkx_labels(G, pos, {n: n for n in abs_words + conc_words}, font_size=7, ax=ax)
    
    ax.set_title(title, fontsize=11)
    if abs_words:
        ax.text(-2, min(pos[w][1] for w in abs_words) - 1.2, 'ABSTRACT',
                ha='center', fontsize=9, color='#d62728', fontweight='bold')
    if conc_words:
        ax.text(2, min(pos[w][1] for w in conc_words) - 1.2, 'CONCRETE',
                ha='center', fontsize=9, color='#2ca02c', fontweight='bold')
    ax.axis('off')

print("Network functions defined.")

In [ ]:
# Select texts with richest data, spanning periods
text_desc_counts = desc_df.dropna(subset=['score']).groupby('_id').size().sort_values(ascending=False)
rich_texts = texts_post[texts_post._id.isin(text_desc_counts.head(30).index)].sort_values('year')

# Sample across periods
selected = []
for period_start in [1470, 1550, 1630, 1700, 1780, 1870, 1950]:
    cands = rich_texts[(rich_texts.year >= period_start) & (rich_texts.year < period_start + 80)]
    if len(cands):
        selected.append(cands.iloc[len(cands)//2])

print(f"Selected {len(selected)} texts for networks:")
for r in selected:
    print(f"  {r.year:.0f}  {r.title[:50]}")

# Plot separate network grids per role
ROLES = ['mod', 'agent', 'patient', 'poss']
ROLE_LABELS = {'mod': 'Modifiers (adjectives)', 'agent': 'Agent verbs (subject)',
               'patient': 'Patient verbs (object)', 'poss': 'Possessives (nouns)'}

for role in ROLES:
    n = len(selected)
    ncols = min(n, 4)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 8 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    fig.suptitle(f'Character ↔ {ROLE_LABELS[role]} networks', fontsize=16, y=1.01)
    
    for idx, row in enumerate(selected):
        text_desc_role = desc_df[(desc_df._id == row['_id']) & (desc_df.role == role)]
        G = build_bimodal_network(text_desc_role, min_char_words=2, top_words=8)
        title = f"{row['title'][:30]} ({row['year']:.0f})"
        plot_bimodal_network(G, title=title, ax=axes[idx])
    
    for idx in range(n, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

## Descriptor word clouds per character

Top abstract and concrete words applied to major characters.

In [ ]:
# Show top abstract/concrete descriptors for major characters across all texts
scored = desc_df.dropna(subset=['score'])

# Top characters by descriptor count
top_chars = scored.groupby(['_id', 'char_name', 'title', 'year']).agg(
    n=('score', 'count'), mean=('score', 'mean')
).reset_index().sort_values('n', ascending=False).head(20)

for _, row in top_chars.iterrows():
    char_desc = scored[(scored._id == row._id) & (scored.char_name == row.char_name)]
    abs_words = char_desc[char_desc.score <= -0.5].word.value_counts().head(8)
    conc_words = char_desc[char_desc.score >= 0.5].word.value_counts().head(8)
    
    print(f"\n{row.char_name} ({row.title[:35]}, {row.year:.0f}) — mean={row['mean']:+.3f}, n={row.n}")
    if len(abs_words):
        print(f"  Abstract: {', '.join(f'{w}({n})' for w, n in abs_words.items())}")
    if len(conc_words):
        print(f"  Concrete: {', '.join(f'{w}({n})' for w, n in conc_words.items())}")

## Quote abstractness: who speaks abstractly?

Compare what characters *say* (quotes) vs how they are *described* (descriptors).

In [ ]:
# Scatter: descriptor score vs quote score per character
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

both = char_profiles_filt.dropna(subset=['desc_mean', 'quote_mean'])
both = both[(both.desc_n >= 10) & (both.quote_n >= 10)]

if len(both):
    # Color by year
    sc = axes[0].scatter(both.desc_mean, both.quote_mean, c=both.year, 
                         cmap='viridis', s=both.desc_n.clip(upper=200), alpha=0.6)
    plt.colorbar(sc, ax=axes[0], label='Year')
    axes[0].set_xlabel('Descriptor concreteness (how described)')
    axes[0].set_ylabel('Quote concreteness (how they speak)')
    axes[0].set_title('Characters: described vs. spoken abstractness')
    axes[0].axhline(0, color='grey', lw=0.5, ls='--')
    axes[0].axvline(0, color='grey', lw=0.5, ls='--')
    
    # Label outliers
    for _, r in both.nlargest(5, 'desc_mean').iterrows():
        axes[0].annotate(r.char_name, (r.desc_mean, r.quote_mean), fontsize=7)
    for _, r in both.nsmallest(5, 'desc_mean').iterrows():
        axes[0].annotate(r.char_name, (r.desc_mean, r.quote_mean), fontsize=7)
    
    corr = both[['desc_mean', 'quote_mean']].corr().iloc[0, 1]
    axes[0].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[0].transAxes, fontsize=12)

# Quote abstractness distribution by period
if len(quotes_df):
    quotes_df['period'] = quotes_df.year.apply(year_to_period)
    period_order = ['C16', 'C17', 'C18', 'C19', 'C20', 'C21']
    period_data = [quotes_df[quotes_df.period == p].mean_score.dropna() for p in period_order]
    period_data = [(p, d) for p, d in zip(period_order, period_data) if len(d) > 0]
    
    if period_data:
        axes[1].boxplot([d for _, d in period_data], labels=[p for p, _ in period_data])
        axes[1].set_xlabel('Period')
        axes[1].set_ylabel('Quote concreteness')
        axes[1].set_title('Distribution of quote abstractness by period')
        axes[1].axhline(0, color='grey', lw=0.5, ls='--')

plt.tight_layout()
plt.show()

## Character abstractness vs text-level abstraction score

Compare the mean abstractness of named characters' descriptors/quotes to the text's overall abstraction score from `data/scores/v8/`.

In [ ]:
# Build per-text comparison: character abstractness vs text-level score
text_char_scores = char_profiles_filt.groupby(['_id', 'year', 'title']).agg(
    char_desc_mean=('desc_mean', 'mean'),
    char_desc_n=('desc_n', 'sum'),
    char_quote_mean=('quote_mean', 'mean'),
).reset_index()

# Look up text-level score (period-matched)
def get_text_score(_id, year):
    if _id not in text_scores:
        return np.nan
    scores = text_scores[_id]
    col = get_norm_col(year)
    # Try period-specific, then median
    if col in scores:
        return scores[col]
    if FALLBACK_COL in scores:
        return scores[FALLBACK_COL]
    # Try any available
    for c, v in scores.items():
        if pd.notna(v):
            return v
    return np.nan

text_char_scores['text_score'] = text_char_scores.apply(
    lambda r: get_text_score(r._id, r.year), axis=1)

matched = text_char_scores.dropna(subset=['text_score', 'char_desc_mean'])
print(f"{len(matched)} texts with both character and text-level scores")
print(f"  (out of {len(text_char_scores)} texts with character data)")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Character descriptor score vs text score
if len(matched) > 2:
    sc = axes[0].scatter(matched.text_score, matched.char_desc_mean, 
                         c=matched.year, cmap='viridis', s=60, alpha=0.7, edgecolors='black', lw=0.5)
    plt.colorbar(sc, ax=axes[0], label='Year')
    
    # Regression line
    from numpy.polynomial.polynomial import polyfit
    m, b = np.polyfit(matched.text_score, matched.char_desc_mean, 1)
    xs = np.linspace(matched.text_score.min(), matched.text_score.max(), 50)
    axes[0].plot(xs, m * xs + b, 'k--', alpha=0.5, lw=1.5)
    
    corr = matched[['text_score', 'char_desc_mean']].corr().iloc[0, 1]
    axes[0].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[0].transAxes, fontsize=12)
    
    # Label points
    for _, r in matched.iterrows():
        axes[0].annotate(r.title[:12], (r.text_score, r.char_desc_mean), 
                         fontsize=6, alpha=0.7, rotation=15)
    
    axes[0].set_xlabel('Text-level abstraction score\n(period-matched vec norms)')
    axes[0].set_ylabel('Mean character descriptor score')
    axes[0].set_title('Character descriptors vs text score')
    axes[0].axhline(0, color='grey', lw=0.5, ls='--')
    axes[0].axvline(0, color='grey', lw=0.5, ls='--')

# 2. Both over time
axes[1].scatter(matched.year, matched.text_score, s=40, alpha=0.6, label='Text score', color='tab:green')
axes[1].scatter(matched.year, matched.char_desc_mean, s=40, alpha=0.6, label='Char descriptors', color='tab:blue')
if len(matched) > 5:
    for col, color in [('text_score', 'tab:green'), ('char_desc_mean', 'tab:blue')]:
        r = matched.sort_values('year').set_index('year')[col].rolling(5, center=True, min_periods=2).mean()
        axes[1].plot(r.index, r.values, color=color, lw=2)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Concreteness score')
axes[1].set_title('Text score vs character descriptor score over time')
axes[1].legend()
axes[1].axhline(0, color='grey', lw=0.5, ls='--')
axes[1].set_xlim(MIN_YEAR - 10, 2030)

# 3. Residual: character score minus text score
matched['residual'] = matched.char_desc_mean - matched.text_score
axes[2].scatter(matched.year, matched.residual, s=40, alpha=0.6, color='tab:purple')
if len(matched) > 5:
    r = matched.sort_values('year').set_index('year')['residual'].rolling(5, center=True, min_periods=2).mean()
    axes[2].plot(r.index, r.values, color='tab:purple', lw=2)
axes[2].axhline(0, color='grey', lw=0.5, ls='--')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Character - Text score')
axes[2].set_title('Residual: are characters more abstract\nthan their text overall?')
axes[2].set_xlim(MIN_YEAR - 10, 2030)
for _, r in matched.iterrows():
    axes[2].annotate(r.title[:12], (r.year, r.residual), fontsize=6, alpha=0.7, rotation=15)

plt.tight_layout()
plt.show()

## Network statistics: abstract/concrete word ratio per text over time

For each text's bimodal network, compute the ratio of abstract to concrete descriptor words.

In [ ]:
# Network-level stats per text
ZCUT = 0.5
net_stats = []
scored = desc_df.dropna(subset=['score'])

for _id in scored._id.unique():
    text_desc = scored[scored._id == _id]
    row = texts_df[texts_df._id == _id].iloc[0]
    
    n_abs = (text_desc.score <= -ZCUT).sum()
    n_conc = (text_desc.score >= ZCUT).sum()
    n_neith = ((text_desc.score > -ZCUT) & (text_desc.score < ZCUT)).sum()
    total = n_abs + n_conc + n_neith
    
    # Unique abstract/concrete words
    abs_words = text_desc[text_desc.score <= -ZCUT].word.nunique()
    conc_words = text_desc[text_desc.score >= ZCUT].word.nunique()
    
    # Per-character: mean abs/conc ratio
    char_stats = text_desc.groupby('char_name').score.agg(['mean', 'count'])
    
    net_stats.append({
        '_id': _id, 'year': row['year'], 'title': row['title'],
        'n_abs': n_abs, 'n_conc': n_conc, 'n_neith': n_neith,
        'pct_abs': n_abs / total if total else 0,
        'pct_conc': n_conc / total if total else 0,
        'abs_conc_ratio': n_abs / n_conc if n_conc else np.nan,
        'unique_abs_words': abs_words, 'unique_conc_words': conc_words,
        'n_chars': char_stats.shape[0],
        'mean_char_score': char_stats['mean'].mean(),
    })

net_df = pd.DataFrame(net_stats).dropna(subset=['year']).sort_values('year')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pct abstract descriptors over time
axes[0].scatter(net_df.year, net_df.pct_abs, s=40, alpha=0.6, label='% abstract')
axes[0].scatter(net_df.year, net_df.pct_conc, s=40, alpha=0.6, label='% concrete')
if len(net_df) > 5:
    for col, color in [('pct_abs', 'tab:blue'), ('pct_conc', 'tab:orange')]:
        r = net_df.set_index('year')[col].rolling(5, center=True, min_periods=2).mean()
        axes[0].plot(r.index, r.values, color=color, lw=2)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Proportion of descriptors')
axes[0].set_title('Abstract vs concrete character descriptors over time')
axes[0].legend()

# Mean character score over time
axes[1].scatter(net_df.year, net_df.mean_char_score, s=40, alpha=0.6)
if len(net_df) > 5:
    r = net_df.set_index('year')['mean_char_score'].rolling(5, center=True, min_periods=2).mean()
    axes[1].plot(r.index, r.values, color='tab:blue', lw=2)
axes[1].axhline(0, color='grey', lw=0.5, ls='--')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Mean character concreteness')
axes[1].set_title('Mean character descriptor concreteness over time')

# Label points
for _, r in net_df.iterrows():
    axes[1].annotate(r.title[:15], (r.year, r.mean_char_score), fontsize=6, alpha=0.7, rotation=20)

plt.tight_layout()
plt.show()